# selfplay_mcts_v1.ipynb — Stage 2: MCTS + ValueNet(input_dim=19)

Stage 2 replaces negamax search with MCTS. The 19 hand-crafted features and
SQLite logging schema are unchanged from Stage 1 (`selfplay_nn.ipynb`).

| Dimension | Stage 1 (selfplay_nn) | Stage 2 (this notebook) |
|---|---|---|
| Search | Negamax depth=3 | MCTS (200 simulations/move) |
| Training signal | TD(0) bootstrap | Game outcome via replay buffer |
| Input | 19 features | 19 features (unchanged) |
| Network | ChessNet MLP 19->32->1 | ValueNet 19->256->128->1 |
| Optimizer | Adam | AdamW + MultiStepLR |

**Training target:** actual game outcome (+1 win / 0 draw / -1 loss) from
the side-to-move perspective at each position -- not TD bootstrapping.

In [ ]:
# Install / imports
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chess", "torch"], check=False)

import os, sqlite3, json, time, random
from collections import deque
from datetime import datetime

import chess
import chess.pgn
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import MultiStepLR

import engine_nn_v2 as eng
from engine_learnable_material import extract_features

FEATURE_NAMES = list(extract_features(chess.Board()).keys())
print(f"Features ({len(FEATURE_NAMES)}): {FEATURE_NAMES}")

In [ ]:
# Configuration
RUN_ID             = 30
N_GAMES            = 200
N_SIMULATIONS      = 200          # MCTS simulations per move
REPLAY_BUFFER_SIZE = 10_000
BATCH_SIZE         = 64
LR                 = 1e-3
WEIGHT_DECAY       = 1e-4
LR_MILESTONES      = [200, 500]   # absolute game counts where LR drops 10x
LR_GAMMA           = 0.1
TEMPERATURE        = 1.0          # sample proportional to visit counts during training
CHECKPOINT_EVERY   = 10
RESUME             = False

DB_PATH    = "../../data/chess_learning.db"
MODEL_DIR  = "../../models"
MODEL_PATH = f"{MODEL_DIR}/mcts_v1_run_{RUN_ID}.pt"

In [ ]:
# Database setup
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

existing = cur.execute("SELECT run_id FROM runs WHERE run_id = ?", (RUN_ID,)).fetchone()
if existing and not RESUME:
    raise RuntimeError(f"run_id={RUN_ID} already exists. Set RESUME=True or choose a new RUN_ID.")
if not existing:
    cur.execute(
        "INSERT INTO runs (run_id, started_at, hyperparameters) VALUES (?, ?, ?)",
        (RUN_ID, datetime.now().isoformat(),
         json.dumps({"n_simulations": N_SIMULATIONS, "lr": LR,
                     "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE,
                     "replay_buffer_size": REPLAY_BUFFER_SIZE,
                     "architecture": "mcts_v1_ValueNet_19features"}))
    )
    con.commit()
    print(f"Created run_id={RUN_ID}")
else:
    print(f"Resuming run_id={RUN_ID}")

row = cur.execute("SELECT MAX(game_number) FROM games WHERE run_id = ?", (RUN_ID,)).fetchone()[0]
start_game = (row or 0) + 1
print(f"Starting at game_number={start_game}")

In [ ]:
# Network, optimizer, replay buffer
os.makedirs(MODEL_DIR, exist_ok=True)

if RESUME and os.path.exists(MODEL_PATH):
    net = eng.load_model(MODEL_PATH, net_class=eng.ValueNet,
                         input_dim=len(FEATURE_NAMES), hidden=256)
    print(f"Loaded model from {MODEL_PATH}")
else:
    net = eng.ValueNet(input_dim=len(FEATURE_NAMES), hidden=256)
    print("Initialized fresh ValueNet(input_dim=19, hidden=256)")

optimizer = optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Session-relative milestones: MultiStepLR counts steps from 0 each session
session_milestones = [max(1, m - start_game + 1) for m in LR_MILESTONES if m >= start_game]
scheduler = MultiStepLR(optimizer, milestones=session_milestones, gamma=LR_GAMMA)
print(f"LR milestones (session-relative): {session_milestones}")

replay_buffer = deque(maxlen=REPLAY_BUFFER_SIZE)

def features_to_tensor(board):
    # Extract 19 features as a float tensor -- used as board_to_input for mcts_search
    return torch.tensor(list(extract_features(board).values()), dtype=torch.float32)

In [ ]:
# Self-play game using MCTS for move selection
def play_game(net, n_simulations=N_SIMULATIONS, temperature=TEMPERATURE):
    # Returns: examples [(feat_tensor, value_target)], board, n_halfmoves
    # value_target is game outcome from the side-to-move perspective (+1/-1/0)
    board     = chess.Board()
    positions = []  # (feat_tensor, board.turn at that position)

    net.eval()
    while not board.is_game_over(claim_draw=True):
        feat = features_to_tensor(board)
        positions.append((feat, board.turn))

        move, _ = eng.mcts_search(
            board, net,
            n_simulations=n_simulations,
            temperature=temperature,
            board_to_input=features_to_tensor,
        )
        board.push(move)

    result    = board.result(claim_draw=True)
    white_val = {"1-0": 1.0, "0-1": -1.0}.get(result, 0.0)

    examples = [
        (feat, white_val if turn == chess.WHITE else -white_val)
        for feat, turn in positions
    ]
    return examples, board, len(positions)

In [ ]:
# Training step -- one gradient update on a replay buffer batch
def train_step(net, optimizer, batch):
    net.train()
    features = torch.stack([x for x, _ in batch])
    targets  = torch.tensor([y for _, y in batch], dtype=torch.float32)

    optimizer.zero_grad()
    loss = F.mse_loss(net(features), targets)
    loss.backward()
    nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
    optimizer.step()
    return loss.item()

In [ ]:
# Logging helpers
def board_to_pgn_str(board, game_number):
    game = chess.pgn.Game.from_board(board)
    game.headers["Event"] = "MCTS v1 Self-Play"
    game.headers["Round"] = str(game_number)
    return str(game)

def log_game(game_number, board, n_halfmoves, duration_s):
    result      = board.result(claim_draw=True)
    outcome_str = {"1-0": "1-0", "0-1": "0-1"}.get(result, "1/2-1/2")
    cur.execute(
        "INSERT INTO games (run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        (RUN_ID, game_number, outcome_str, n_halfmoves, round(duration_s, 2), "natural")
    )
    game_id = cur.lastrowid
    cur.execute(
        "INSERT INTO game_records (game_id, pgn) VALUES (?, ?)",
        (game_id, board_to_pgn_str(board, game_number))
    )
    con.commit()
    return game_id, outcome_str

def log_weights(game_number, prev_weights):
    attr = eng.attribution_weights(
        chess.Board(), net,
        board_to_input=features_to_tensor,
        feature_names=FEATURE_NAMES,
    )
    for name, val in attr.items():
        cur.execute(
            "INSERT INTO weights (run_id, game_number, feature_name, weight_value) "
            "VALUES (?,?,?,?)",
            (RUN_ID, game_number, name, round(float(val), 6))
        )
        if prev_weights and name in prev_weights:
            cur.execute(
                "INSERT INTO weight_deltas (run_id, game_number, feature_name, delta) "
                "VALUES (?,?,?,?)",
                (RUN_ID, game_number, name, round(float(val) - prev_weights[name], 6))
            )
    con.commit()
    return attr

def log_feature_stats(game_id, examples):
    if not examples:
        return
    means = torch.stack([x for x, _ in examples]).mean(dim=0).tolist()
    cur.executemany(
        "INSERT INTO feature_stats (game_id, feature_name, mean_value) VALUES (?,?,?)",
        [(game_id, name, round(means[i], 6)) for i, name in enumerate(FEATURE_NAMES)]
    )
    con.commit()

In [ ]:
# Training loop
prev_weights   = None
decisive_count = 0

print(f"MCTS v1  RUN_ID={RUN_ID}  simulations={N_SIMULATIONS}  "
      f"games {start_game}-{start_game + N_GAMES - 1}")
print("-" * 72)

for game_idx in range(N_GAMES):
    game_number = start_game + game_idx
    t0 = time.time()

    examples, board, n_halfmoves = play_game(net)
    replay_buffer.extend(examples)
    duration = time.time() - t0

    loss = None
    if len(replay_buffer) >= BATCH_SIZE:
        batch = random.sample(replay_buffer, BATCH_SIZE)
        loss  = train_step(net, optimizer, batch)
    scheduler.step()

    game_id, outcome_str = log_game(game_number, board, n_halfmoves, duration)
    log_feature_stats(game_id, examples)
    if outcome_str != "1/2-1/2":
        decisive_count += 1

    if game_number % CHECKPOINT_EVERY == 0:
        prev_weights = log_weights(game_number, prev_weights)
        eng.save_model(net, MODEL_PATH)

    loss_str     = f"{loss:.4f}" if loss is not None else "  n/a "
    decisive_pct = decisive_count / (game_idx + 1) * 100
    print(f"G{game_number:4d} | {outcome_str:7s} | {n_halfmoves:3d} ply | "
          f"{duration:5.1f}s | loss={loss_str} | "
          f"buf={len(replay_buffer):5d} | decisive={decisive_pct:4.1f}%")

print("-" * 72)
print(f"Done. Model saved to {MODEL_PATH}")
con.close()